# Insurance Renewal — Complete Analysis Notebook

This notebook contains EDA, feature engineering, model training, interpretability (SHAP), comprehensive evaluation, and reporting. Run cells sequentially. Heavy cells are guarded.

In [ ]:

# 1. Imports & environment
import os, time, warnings, random
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set(style='whitegrid')
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Make dirs
os.makedirs('Visualizations', exist_ok=True)
os.makedirs('Visualizations/Results', exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Environment ready.')


In [ ]:

# 2. Load data
DATA_PATH = 'Dataset/train_ZoGVYWq.csv'
if os.path.exists(DATA_PATH):
    df_raw = pd.read_csv(DATA_PATH)
    print('Loaded:', df_raw.shape)
else:
    df_raw = None
    print('Dataset not found:', DATA_PATH)


## 3. Exploratory Data Analysis (EDA)
Includes missing value heatmap, target distribution, numeric distributions, categorical counts, correlation heatmap.

In [ ]:

# EDA: quick overview + missing values heatmap
if df_raw is None:
    print('No data for EDA.')
else:
    display(df_raw.head())
    print('\nColumns:', df_raw.columns.tolist())
    print('\nMissing values per column:')
    print(df_raw.isnull().sum().sort_values(ascending=False).head(20))
    # Missing heatmap (sample if large)
    try:
        plt.figure(figsize=(10,6))
        sns.heatmap(df_raw.isnull(), cbar=False, cmap='viridis')
        plt.title('Missing Value Heatmap')
        plt.show()
    except Exception as e:
        print('Missing-heatmap failed:', e)


In [ ]:

# Target distribution and class imbalance
if df_raw is not None and 'renewal' in df_raw.columns:
    print('Target distribution:')
    print(df_raw['renewal'].value_counts(dropna=False))
    plt.figure(figsize=(5,3))
    sns.countplot(x='renewal', data=df_raw)
    plt.title('Target Distribution (renewal)')
    plt.show()
else:
    print('renewal column not present.')    


In [ ]:

# Numeric summary and histograms for selected columns (safe fallback list)
candidate_cols = ['perc_premium_paid_by_cash_credit','age_in_days','Income','no_of_premiums_paid','premium','application_underwriting_score']
num_cols = [c for c in candidate_cols if df_raw is not None and c in df_raw.columns]
if df_raw is not None and num_cols:
    display(df_raw[num_cols].describe().round(3))
    df_raw[num_cols].hist(bins=20, figsize=(12,6))
    plt.suptitle('Distribution of key numeric features')
    plt.show()
else:
    print('No numeric columns from candidate list found for plotting.')


In [ ]:

# Categorical distributions
cat_cols = [c for c in ['sourcing_channel','residence_area_type','payment_method','product_type'] if df_raw is not None and c in df_raw.columns]
for col in cat_cols:
    plt.figure(figsize=(6,3))
    sns.countplot(x=col, data=df_raw, order=df_raw[col].value_counts().index[:20])
    plt.title(f'{col} distribution')
    plt.xticks(rotation=45)
    plt.show()


## 4. Data cleaning & feature engineering (canonical)

In [ ]:

# Cleaning + core feature engineering
if df_raw is None:
    df = None
else:
    df = df_raw.copy()
    # Fill late payment NaNs with 0
    late_cols = ['Count_3-6_months_late','Count_6-12_months_late','Count_more_than_12_months_late']
    for c in late_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
    # Drop rows with missing underwriting score (small fraction)
    if 'application_underwriting_score' in df.columns:
        df = df.dropna(subset=['application_underwriting_score']).reset_index(drop=True)
    # Create derived features
    if 'age_in_days' in df.columns and 'age_years' not in df.columns:
        df['age_years'] = df['age_in_days'] / 365.25
    if 'Income' in df.columns and 'income_log' not in df.columns:
        df['income_log'] = np.log1p(df['Income'])
    if all(c in df.columns for c in late_cols):
        df['total_late_counts'] = df[late_cols].sum(axis=1)
    if 'no_of_premiums_paid' in df.columns:
        df['no_of_premiums_paid'] = pd.to_numeric(df['no_of_premiums_paid'], errors='coerce').fillna(0)
        df['late_rate'] = df['total_late_counts'] / df['no_of_premiums_paid'].replace(0, np.nan)
        df['late_rate'] = df['late_rate'].fillna(0)
    if 'total_late_counts' in df.columns:
        df['any_late'] = (df['total_late_counts'] > 0).astype(int)
    if 'premium' in df.columns and 'Income' in df.columns:
        df['premium_to_income'] = df['premium'] / (df['Income'] + 1)
    # High underwriting flag
    if 'application_underwriting_score' in df.columns:
        thr = df['application_underwriting_score'].quantile(0.99)
        df['high_underwriting_flag'] = (df['application_underwriting_score'] >= thr).astype(int)
    # Print some checks
    print('After feature engineering, shape:', df.shape)
    display(df[['age_years','income_log','total_late_counts','late_rate','any_late']].head())


## 5. Per-feature analysis (descriptive & by-target)

In [ ]:

# Per-feature analysis examples (numeric)
if df is not None:
    features = ['perc_premium_paid_by_cash_credit','age_years','Income','total_late_counts','late_rate','application_underwriting_score','premium']
    for f in features:
        if f in df.columns:
            print('\nFeature:', f)
            print(df[f].describe().round(3))
            if 'renewal' in df.columns:
                print(df.groupby('renewal')[f].describe().round(3))
else:
    print('No df for per-feature analysis.')


In [ ]:

# Categorical per-feature analysis
if df is not None:
    for c in ['sourcing_channel','residence_area_type']:
        if c in df.columns:
            print('\n', c, 'value counts:')
            print(df[c].value_counts())
            if 'renewal' in df.columns:
                print(df.groupby(c)['renewal'].mean().round(3))
else:
    print('No df for categorical analysis.')


## 6. Visualization-driven EDA (distributions, boxplots, correlation)

In [ ]:

# Boxplots of key numeric features by target
if df is not None and 'renewal' in df.columns:
    cols = [c for c in ['perc_premium_paid_by_cash_credit','age_years','Income','no_of_premiums_paid','premium','application_underwriting_score','late_rate'] if c in df.columns]
    n = len(cols)
    plt.figure(figsize=(5*n if n<4 else 15, 4* ((n+2)//3)))
    for i,c in enumerate(cols,1):
        plt.subplot((n+2)//3,3,i)
        sns.boxplot(x='renewal', y=c, data=df)
        plt.title(c)
    plt.tight_layout()
    plt.show()


In [ ]:

# Correlation heatmap (numeric features)
if df is not None:
    corr_cols = [c for c in ['perc_premium_paid_by_cash_credit','age_years','Income','no_of_premiums_paid','premium','application_underwriting_score','late_rate','total_late_counts'] if c in df.columns]
    if len(corr_cols)>1:
        plt.figure(figsize=(10,7))
        sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
        plt.title('Correlation Heatmap (selected numeric features)')
        plt.show()


## 7. Consolidated Feature Engineering & Preprocessor (pipeline)

In [ ]:

# Build final feature list and preprocessor
if df is not None:
    numeric_features = [c for c in ['age_years','income_log','premium','total_late_counts','late_rate','perc_premium_paid_by_cash_credit','application_underwriting_score','premium_to_income'] if c in df.columns]
    categorical_features = [c for c in ['sourcing_channel','residence_area_type'] if c in df.columns]
    print('Numeric features:', numeric_features)
    print('Categorical features:', categorical_features)
    # OneHotEncoder compatibility wrapper
    from sklearn import __version__ as skv
    from packaging import version
    if version.parse(skv) >= version.parse("1.2"):
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    else:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)
    num_transformer = Pipeline([('scaler', StandardScaler())])
    cat_transformer = Pipeline([('onehot', ohe)])
    preprocessor = ColumnTransformer(transformers=[
        ('num', num_transformer, numeric_features),
        ('cat', cat_transformer, categorical_features)
    ], remainder='drop')
else:
    numeric_features = categorical_features = preprocessor = None
    print('No df available to build preprocessor.')


## 8. Train / Test Split (for modeling)

In [ ]:

# Train/test split (keep outside earlier split to ensure updated df)
if df is not None:
    target = 'renewal'
    X = df.copy()
    y = X.pop(target)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    print('Train/test shapes:', X_train.shape, X_test.shape)
else:
    print('No df available.')


## 9. Modeling — training concrete models (guarded)
Includes: Logistic Regression, XGBoost (RandomizedSearch), Neural Net (Keras), BalancedRF, EasyEnsemble, TabNet (guarded).

In [ ]:

# 9a) Model training implementations (safe, with try/except guards)
training_times = {}

# Logistic Regression
try:
    from sklearn.linear_model import LogisticRegression
    print('\nTraining Logistic Regression...')
    pipe_lr = Pipeline([('preproc', preprocessor), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))])
    t0 = time.time()
    pipe_lr.fit(X_train, y_train)
    training_times['Logistic Regression'] = time.time()-t0
    preds_lr = pipe_lr.predict(X_test)
    probs_lr = pipe_lr.predict_proba(X_test)[:,1]
    save_path = os.path.join('Visualizations/Results','lr_predictions.csv')
    pd.DataFrame({'y_true': y_test, 'y_pred': preds_lr, 'prob': probs_lr}).to_csv(save_path, index=False)
    print('Saved LR preds to', save_path)
except Exception as e:
    print('LR training skipped/failed:', e)


In [ ]:

# 9b) XGBoost with RandomizedSearchCV
try:
    import xgboost as xgb
    from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
    print('\nTraining XGBoost (light RandomizedSearchCV)...')
    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_jobs=-1, random_state=SEED)
    pipe_xgb = Pipeline([('preproc', preprocessor), ('clf', xgb_clf)])
    param_dist = {
        'clf__n_estimators': [100,200],
        'clf__max_depth': [3,5],
        'clf__learning_rate': [0.01,0.1],
        'clf__subsample': [0.8,1.0],
        'clf__colsample_bytree': [0.8,1.0]
    }
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    search = RandomizedSearchCV(pipe_xgb, param_distributions=param_dist, n_iter=6, scoring='roc_auc', cv=cv, verbose=1, random_state=SEED, n_jobs=-1)
    t0 = time.time()
    search.fit(X_train, y_train)
    training_times['XGBoost'] = time.time()-t0
    best_xgb = search.best_estimator_
    preds_xgb = best_xgb.predict(X_test)
    probs_xgb = best_xgb.predict_proba(X_test)[:,1]
    pd.DataFrame({'y_true': y_test, 'y_pred': preds_xgb, 'prob': probs_xgb}).to_csv('Visualizations/Results/xgb_predictions.csv', index=False)
    print('XGBoost saved preds; best params:', search.best_params_)
except Exception as e:
    print('XGBoost skipped/failed:', e)


In [ ]:

# 9c) Neural Network (Keras)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print('\nTraining Neural Network (Keras)...')
    # Fit preprocessor once (for NN we need dense array)
    preprocessor.fit(X_train)
    X_train_trans = preprocessor.transform(X_train)
    X_test_trans = preprocessor.transform(X_test)
    input_dim = X_train_trans.shape[1]
    nn_model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])
    nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    t0 = time.time()
    history = nn_model.fit(X_train_trans, y_train, validation_split=0.15, epochs=15, batch_size=64, verbose=1)
    training_times['Neural Network'] = time.time()-t0
    probs_nn = nn_model.predict(X_test_trans).ravel()
    preds_nn = (probs_nn>=0.5).astype(int)
    pd.DataFrame({'y_true': y_test, 'y_pred': preds_nn, 'prob': probs_nn}).to_csv('Visualizations/Results/nn_predictions.csv', index=False)
    print('Neural Net saved preds.')
except Exception as e:
    print('Neural Net skipped/failed:', e)


In [ ]:

# 9d) BalancedRandomForest & EasyEnsemble (imblearn)
try:
    from imblearn.ensemble import BalancedRandomForestClassifier, EasyEnsembleClassifier
    print('\nTraining Balanced Random Forest...')
    brf = BalancedRandomForestClassifier(n_estimators=100, random_state=SEED)
    t0 = time.time()
    brf.fit(preprocessor.transform(X_train), y_train)
    training_times['Balanced RF'] = time.time()-t0
    probs_brf = brf.predict_proba(preprocessor.transform(X_test))[:,1]
    preds_brf = (probs_brf>=0.5).astype(int)
    pd.DataFrame({'y_true': y_test, 'y_pred': preds_brf, 'prob': probs_brf}).to_csv('Visualizations/Results/balanced_rf_predictions.csv', index=False)
    print('Balanced RF saved preds.')
    print('\nTraining EasyEnsemble...')
    eec = EasyEnsembleClassifier(n_estimators=10, random_state=SEED)
    t0 = time.time()
    eec.fit(preprocessor.transform(X_train), y_train)
    training_times['Easy Ensemble'] = time.time()-t0
    probs_eec = eec.predict_proba(preprocessor.transform(X_test))[:,1]
    preds_eec = (probs_eec>=0.5).astype(int)
    pd.DataFrame({'y_true': y_test, 'y_pred': preds_eec, 'prob': probs_eec}).to_csv('Visualizations/Results/easy_ensemble_predictions.csv', index=False)
    print('Easy Ensemble saved preds.')
except Exception as e:
    print('Imbalance ensembles skipped/failed:', e)


In [ ]:

# 9e) TabNet (guarded; requires pytorch_tabnet)
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    print('\nTraining TabNet (guarded)...')
    X_train_tab = preprocessor.transform(X_train)
    X_test_tab = preprocessor.transform(X_test)
    import numpy as _np
    X_train_tab = _np.asarray(X_train_tab)
    X_test_tab = _np.asarray(X_test_tab)
    tabnet_model = TabNetClassifier(n_d=8, n_a=8, n_steps=3, verbose=0)
    t0 = time.time()
    tabnet_model.fit(X_train_tab, y_train.values, eval_set=[(X_test_tab, y_test.values)], max_epochs=50, patience=10, batch_size=1024)
    training_times['TabNet'] = time.time()-t0
    y_pred_tabnet = tabnet_model.predict(X_test_tab)
    y_pred_proba_tabnet = tabnet_model.predict_proba(X_test_tab)
    pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_tabnet, 'prob': y_pred_proba_tabnet[:,1]}).to_csv('Visualizations/Results/tabnet_predictions.csv', index=False)
    print('TabNet saved preds.')
except Exception as e:
    print('TabNet skipped/failed:', e)


## 10. Model Evaluation Visualizations
ROC, PR, Confusion Matrices, Calibration, Precision-Recall comparison, Prediction distribution.

In [ ]:

# Load predictions and compute metrics + plots
from sklearn.metrics import roc_curve, precision_recall_curve, auc, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score
pred_files = {
    'LR': 'Visualizations/Results/lr_predictions.csv',
    'XGB': 'Visualizations/Results/xgb_predictions.csv',
    'NN': 'Visualizations/Results/nn_predictions.csv',
    'BRF': 'Visualizations/Results/balanced_rf_predictions.csv',
    'EEC': 'Visualizations/Results/easy_ensemble_predictions.csv',
    'TAB': 'Visualizations/Results/tabnet_predictions.csv'
}

results = {}
for name, path in pred_files.items():
    if os.path.exists(path):
        dfp = pd.read_csv(path)
        y_true = dfp['y_true'].values
        y_pred = dfp['y_pred'].values if 'y_pred' in dfp.columns else (dfp['prob'].values>=0.5).astype(int)
        prob = dfp['prob'].values if 'prob' in dfp.columns else None
        results[name] = {'y_true': y_true, 'y_pred': y_pred, 'prob': prob}
        print(f'Loaded {name} predictions, n={len(y_true)}')
    else:
        print(f'{name} preds not found ({path}) — skipping.')

# Plot ROC and PR curves for available models
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
for name, d in results.items():
    if d['prob'] is not None:
        fpr, tpr, _ = roc_curve(d['y_true'], d['prob'])
        plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(d["y_true"], d["prob"]):.3f})')
plt.plot([0,1],[0,1],'k--',alpha=0.5)
plt.title('ROC Curves')
plt.legend()

plt.subplot(1,2,2)
for name, d in results.items():
    if d['prob'] is not None:
        prec, rec, _ = precision_recall_curve(d['y_true'], d['prob'])
        ap = average_precision_score(d['y_true'], d['prob'])
        plt.plot(rec, prec, label=f'{name} (AP={ap:.3f})')
plt.title('Precision-Recall Curves')
plt.legend()
plt.tight_layout()
plt.show()

# Confusion matrices
for name, d in results.items():
    cm = confusion_matrix(d['y_true'], d['y_pred'])
    disp = ConfusionMatrixDisplay(cm)
    plt.figure(figsize=(4,3))
    disp.plot(cmap='Blues', colorbar=False)
    plt.title(f'Confusion Matrix - {name}')
    plt.show()


## 11. Feature Importance & SHAP analysis (mapping + plots)

In [ ]:

# Map XGBoost importances to feature names (if available)
try:
    # try to extract from best_xgb pipeline
    pipe = globals().get('best_xgb', globals().get('pipe_xgb', None))
    if pipe is None:
        pipe = globals().get('best_xgb', None)
    if pipe is None and 'search' in globals():
        # fallback to search.best_estimator_ if name differs
        try:
            pipe = globals()['search'].best_estimator_
        except Exception:
            pass
    if pipe is not None and hasattr(pipe, 'named_steps'):
        clf = pipe.named_steps.get('clf', pipe)
    else:
        clf = pipe
    if clf is not None and hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
        feat_names = get_feature_names_from_preprocessor(preprocessor, numeric_features, categorical_features)
        if feat_names is None or len(feat_names)!=len(importances):
            # fallback to X_train columns
            feat_names = X_train.columns.tolist()
        fi_df = pd.DataFrame({'feature': feat_names, 'importance': importances}).sort_values('importance', ascending=False)
        fi_df.to_csv('Visualizations/Results/xgb_feature_importances_mapped.csv', index=False)
        display(fi_df.head(20))
    else:
        print('XGBoost classifier or feature_importances_ not found.')
except Exception as e:
    print('Mapping XGBoost importances failed:', e)


In [ ]:

# SHAP (if available) - use small sample to avoid memory issues
try:
    import shap
    shap_ok = True
    print('SHAP version:', shap.__version__)
except Exception as e:
    shap_ok = False
    print('SHAP not available:', e)

if shap_ok and 'best_xgb' in globals():
    try:
        model_for_shap = globals()['best_xgb'].named_steps['clf'] if hasattr(globals()['best_xgb'], 'named_steps') else globals()['best_xgb']
        X_test_trans = preprocessor.transform(X_test)
        n_sample = min(2000, X_test_trans.shape[0])
        idx = np.random.choice(X_test_trans.shape[0], n_sample, replace=False)
        X_shap = X_test_trans[idx]
        expl = shap.TreeExplainer(model_for_shap)
        shap_values = expl.shap_values(X_shap) if hasattr(expl, 'shap_values') else expl(X_shap)
        # plots (save to files)
        try:
            plt.figure(figsize=(8,6)); shap.summary_plot(shap_values, features=X_shap, show=False); plt.tight_layout(); plt.savefig('Visualizations/shap_summary_plot.png'); plt.close()
            plt.figure(figsize=(8,6)); shap.summary_plot(shap_values, features=X_shap, plot_type='bar', show=False); plt.tight_layout(); plt.savefig('Visualizations/shap_bar_plot.png'); plt.close()
            print('Saved SHAP plots to Visualizations/')
        except Exception as e:
            print('SHAP plotting failed:', e)
    except Exception as e:
        print('SHAP computation failed:', e)
else:
    print('Skipping SHAP (not available or model missing).')


## 12. Error Analysis & Segment Performance

In [ ]:

# Error analysis: examine false positives/negatives with respect to premium and claim ratio (if available)
if os.path.exists('Visualizations/Results/xgb_predictions.csv'):
    dfp = pd.read_csv('Visualizations/Results/xgb_predictions.csv')
    merged = X_test.reset_index(drop=True).copy()
    merged['y_true'] = dfp['y_true']
    merged['y_pred'] = dfp['y_pred']
    merged['error_type'] = merged.apply(lambda r: 'TP' if r['y_true']==1 and r['y_pred']==1 else ('TN' if r['y_true']==0 and r['y_pred']==0 else ('FP' if r['y_pred']==1 else 'FN')), axis=1)
    # Example error grouping by premium increase or claim_ratio if present
    for feat in ['premium_rate_increase','claim_ratio','premium']:
        if feat in merged.columns:
            plt.figure(figsize=(6,3))
            sns.boxplot(x='error_type', y=feat, data=merged)
            plt.title(f'Error type by {feat} (XGBoost)')
            plt.show()
else:
    print('XGBoost predictions not found for error analysis.')


## 13. Model Comparison Table & Summary Metrics

In [ ]:

# Build comparison table from saved metrics and predictions
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
models = {}
for name, path in [('LR','Visualizations/Results/lr_predictions.csv'),
                   ('XGB','Visualizations/Results/xgb_predictions.csv'),
                   ('NN','Visualizations/Results/nn_predictions.csv'),
                   ('BRF','Visualizations/Results/balanced_rf_predictions.csv'),
                   ('EEC','Visualizations/Results/easy_ensemble_predictions.csv'),
                   ('TAB','Visualizations/Results/tabnet_predictions.csv')]:
    if os.path.exists(path):
        dfp = pd.read_csv(path)
        y_true = dfp['y_true'].values
        prob = dfp['prob'].values if 'prob' in dfp.columns else None
        pred = dfp['y_pred'].values if 'y_pred' in dfp.columns else (prob>=0.5).astype(int)
        metrics = {
            'Accuracy': accuracy_score(y_true, pred),
            'F1': f1_score(y_true, pred),
            'Precision': precision_score(y_true, pred),
            'Recall': recall_score(y_true, pred),
            'ROC AUC': roc_auc_score(y_true, prob) if prob is not None else None,
            'PR AUC': average_precision_score(y_true, prob) if prob is not None else None
        }
        models[name] = metrics
# Display
if models:
    comp_df = pd.DataFrame(models).T
    display(comp_df.round(4))
    comp_df.to_csv('Visualizations/Results/model_comparison_metrics.csv')
    print('Saved model comparison metrics to Visualizations/Results/model_comparison_metrics.csv')
else:
    print('No model prediction files found to compare.')


## 14. Final Reporting & Recommendations

In [ ]:

# Write a short markdown report summarizing key artifacts and instructions
report_path = os.path.join('Visualizations','report.md')
with open(report_path, 'w') as f:
    f.write('# Insurance Renewal Analysis Report\n\n')
    f.write('## Generated artifacts\n\n')
    for root, dirs, files in os.walk('Visualizations'):
        for fn in files:
            f.write(f'- {os.path.join(root, fn)}\n')
    f.write('\n## Notes\n- Run heavy cells (XGBoost, NN, TabNet) after verifying the EDA and preprocessing.\n')
print('Wrote report to', report_path)
